# Lab 3: From Random Numbers to Random Variables

Complete the code TODOs, the required written responses, the manager recommendation, and the AI-use statement. Think about the discussion questions and be prepared to explain your reasoning to the TA. Every code location that you must complete is marked `TODO`; code marked **Provided** or **do not edit** should be left unchanged.

## Student Information

**Name:** Write your answer here  
**Student ID:** Write your answer here  
**Date:** Write your answer here

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Part A: Generate Exponential Service Times

Use `mean_service = 3.0` and the inverse-transform formula:

```text
lambda = 1 / mean_service
X = -log(1 - U) / lambda
```

In [ ]:
rng = np.random.default_rng(211)
mean_service = 3.0
lambda_rate = 1 / mean_service
n = 1000

# Generate n uniform random numbers.
u = TODO 

# Apply the inverse-transform formula.
service_inverse = TODO

# Check that all service times are non-negative.
all_non_negative = TODO

# Provided summary-table code -- do not edit.
inverse_summary = pd.DataFrame({
    "Mean": [service_inverse.mean()],
    "Standard deviation": [service_inverse.std(ddof=1)],
    "Minimum": [service_inverse.min()],
    "Maximum": [service_inverse.max()],
    "All non-negative?": [all_non_negative]
}, index=["Inverse transform"])

inverse_summary

In [ ]:
# Provided plotting code -- do not edit.
plt.hist(service_inverse, bins=35, edgecolor="black")
plt.xlabel("Service time (minutes)")
plt.ylabel("Frequency")
plt.title("Inverse-transform exponential service times")
plt.show()

### Part A Questions

Think about the following questions:

1. Why are most service times short while a few are long?
2. Why can a long service time matter in a queue?
3. Why does this method start with uniform random numbers?

## Part B: Confirm Against NumPy's Generator

Generate 1,000 exponential service times with `rng.exponential` and confirm they are
consistent with the manual inverse-transform sample from Part A. A provided check does the
comparison for you — if it fails, the bug is almost always in the mean-vs-rate argument.

In [ ]:
rng = np.random.default_rng(211)

# Generate n exponential service times with rng.exponential.
# Reminder: rng.exponential takes the MEAN as its argument, not the rate lambda.
service_numpy = TODO

# Provided check -- do not edit.
assert service_inverse.size == n and service_numpy.size == n, \
    "Both samples must have length n before running this check."
print(f"Inverse-transform sample mean : {service_inverse.mean():.4f}")
print(f"NumPy sample mean             : {service_numpy.mean():.4f}")
print(f"Model mean (mean_service)     : {mean_service:.4f}")
assert np.isclose(service_inverse.mean(), service_numpy.mean(), rtol=0.15), (
    "The two sample means differ by more than expected sampling noise -- "
    "check your rng.exponential call (mean vs rate) and your Part A formula."
)
print("Check passed: both samples are consistent with Exponential(mean=3.0).")

## Part C: Use Accept-Reject Sampling

Generate 1,000 values from `f(x) = 2x` on `[0, 1]`. Propose `x` and `u` independently from `Uniform(0, 1)` and accept `x` when `u <= x`.

In [ ]:
rng = np.random.default_rng(211)
target_accepts = 1000
accepted = []
proposals = 0

# Write a while loop below to collect target_accepts accepted values:
# 1. generate x and u
# 2. increase proposals
# 3. append x when u <= x
TODO

accepted = np.asarray(accepted)

# Calculate the acceptance rate and sample mean.
acceptance_rate = TODO
accepted_mean = TODO

pd.DataFrame({
    "Accepted values": [len(accepted)],
    "Proposals": [proposals],
    "Acceptance rate": [acceptance_rate],
    "Sample mean": [accepted_mean]
})

In [ ]:
# Provided plotting code -- do not edit.
plt.hist(accepted, bins=30, range=(0, 1), edgecolor="black")
plt.xlabel("x")
plt.ylabel("Frequency")
plt.title("Accept-reject samples for f(x) = 2x")
plt.show()

### Part C Questions

Think about the following questions:

1. Why should the histogram contain more values near 1 than near 0?
2. Why are rejected proposals not coding errors?
3. What feature of an accept-reject method determines its efficiency?

## Checkpoint 1: Sampling Methods (1 of 3 Points)

Target time: approximately 60 minutes into the lab.

Show your notebook to the TA after completing Parts A-C. To receive the first 1 point, your notebook must contain:

1. the Part A inverse-transform sample, non-negative check, summary table, and histogram;
2. the Part B NumPy sample, passed consistency check, and written response; and
3. the Part C accept-reject sample, proposal count, acceptance rate, sample mean, and histogram.

Be prepared to explain one of the sampling methods. Continue with Parts D-E after the checkpoint.

## Part D: Simulate Drink Orders and Service Times

Simulate 1,000 drink choices, then generate an exponential service time using the mean for
each customer's selected drink.

**Use the discrete inverse-transform method.** That is:

1. Build the cumulative probabilities (the discrete CDF) for the four drinks.
2. Generate one uniform value per customer for the drink-choice step.
3. For each uniform value, find the smallest cut point that is `>=` that value — this is
   exactly `np.searchsorted(cut_points, u, side="left")`.
4. Generate each customer's service time only *after* their drink is assigned, using the
   Part A inverse-transform formula with that drink's mean.

In [ ]:
drinks = np.array(["Coffee", "Tea", "Chocolate", "Matcha"])
baseline_probabilities = np.array([0.35, 0.20, 0.30, 0.15])
mean_service_by_drink = {
    "Coffee": 2.0,
    "Tea": 1.5,
    "Chocolate": 4.0,
    "Matcha": 5.0,
}
n_customers = 1000
rng = np.random.default_rng(211)

# Build the cumulative probability (discrete CDF) cut points for the four drinks,
# in the order given by `drinks` / `baseline_probabilities`.
baseline_cut_points = TODO

# Generate n_customers uniform random numbers for the drink-choice step.
choice_u = TODO

# Apply the discrete inverse-transform rule: find the smallest cut point >= each
# uniform value, then look up the corresponding drink. Clip the index so a uniform value
# of exactly 1.0 cannot fall outside the array.
baseline_drink_index = TODO
baseline_orders = TODO

# Generate n_customers uniform random numbers for the service-time step.
# (draw these AFTER choice_u, from the same rng, so drink is assigned before service time)
service_u = TODO

# Look up each customer's drink-specific mean, then generate their service time
# with the Part A inverse-transform formula
baseline_means = TODO
baseline_service = TODO

baseline_data = pd.DataFrame({
    "Drink": baseline_orders,
    "Service time": baseline_service,
})
baseline_data.head()

In [ ]:
# Provided summary-table code -- do not edit.
baseline_summary = (
    baseline_data.groupby("Drink")
    .agg(
        Count=("Drink", "size"),
        Average_service_time=("Service time", "mean"),
    )
    .reindex(drinks)
)
baseline_summary["Simulated proportion"] = (
    baseline_summary["Count"] / n_customers
)
baseline_summary = baseline_summary[[
    "Count", "Simulated proportion", "Average_service_time"
]]
baseline_summary.round(4)

In [ ]:
# Calculate the overall average and longest service times.
baseline_average = TODO
baseline_longest = TODO

pd.DataFrame({
    "Overall average service time": [baseline_average],
    "Longest service time": [baseline_longest],
})

### Part D Questions

#### Expected workload contribution

Let $D$ be the drink category and $T$ the service time. For drink $j$, write

$$p_j=P(D=j), \qquad \mu_j=E[T\mid D=j].$$

A drink can create high workload because it is ordered frequently, takes a long time to prepare, or both. Define its expected contribution to service demand per arriving customer as

$$p_j\mu_j=P(D=j)E[T\mid D=j].$$

> **Important distinction:** $\mu_j$ is the mean service time **if drink $j$ is ordered**, whereas $p_j\mu_j$ is drink $j$'s expected contribution to service demand **per arriving customer**.

The category contributions add to the overall expected service time:

$$E[T]=\sum_j p_j\mu_j.$$

Think about the following questions:

1. Why do the simulated proportions not exactly equal the model probabilities?
2. Which drink contributes the most to expected workload? Consider both how frequently it is ordered and its mean service time. Explain using $p_j\mu_j$.
3. Is this necessarily the same drink as the one with the largest conditional mean $\mu_j$? Explain.
4. Why should drink choice be generated before service time?
5. The continuous inverse-transform proof from §5.2 relies on `F` being invertible. Why does
   that proof not apply directly to `baseline_cut_points`, and what did the discrete method
   above do instead to work around it?

## Part E: Reflect on a Matcha Trend

No new simulation is required for this part. Suppose matcha's probability rises from `0.15` to `0.35`. The other three probabilities decrease in the same proportion, so the new probabilities still sum to 1. The drink-specific service-time means stay unchanged.

| Drink | Baseline probability | Matcha-trend probability |
|---|---:|---:|
| Coffee | 0.35 | $0.35(0.65/0.85) \approx 0.2676$ |
| Tea | 0.20 | $0.20(0.65/0.85) \approx 0.1529$ |
| Chocolate | 0.30 | $0.30(0.65/0.85) \approx 0.2294$ |
| Matcha | 0.15 | 0.35 |

Part D defined each category's expected workload contribution as $p_j\mu_j$. Adding those contributions gives the Law of Total Expectation:

$$\underbrace{p_j\mu_j}_{\text{contribution from category }j}
\quad\Longrightarrow\quad
\underbrace{\sum_j p_j\mu_j}_{E[T]}.$$

Once $p_j$ and $\mu_j$ are specified, this expected service demand can be calculated exactly from the model. No new random sample is needed: simulate when necessary; calculate analytically when the model gives an exact quantity.

Answer the following without generating another random sample.

1. Calculate the expected service time under the baseline and matcha-trend probabilities. Which direction does the expected workload move, and why?

   **Response 1:** Write your answer here.

2. Does the longest observed service time in a 1,000-customer sample necessarily move in the same direction as the expected service time? Why or why not?

   **Response 2:** Write your answer here.

3. What should a manager check in real data before acting on this predicted shift?

   **Response 3:** Write your answer here.

**Manager recommendation (2-4 sentences):** Write your answer here.

**AI-use statement:** Write your answer here.

## Checkpoint 2: Final Submission

Show the completed Part D (simulation, summary table, overall average/longest service
times), your Part E written reflection, manager recommendation, and AI-use statement to the
TA. Be prepared to explain, in terms of $E[T]=\sum_j p_j\mu_j$, why the matcha trend changes
the predicted workload. After the final checkpoint:

1. save the notebook as `Lab03_StudentID.ipynb`, for example `Lab03_6900789.ipynb`;
2. restart the kernel and run all cells from top to bottom;
3. check that there are no errors;
4. save the notebook; and
5. upload the single completed `.ipynb` file to the **Lab 03** assignment in Google Classroom before leaving the lab.